In [ ]:
#author july 


In [8]:
#EW+NS=vv
#2025.09.18
import matplotlib.pyplot as plt
import numpy as np
import os  

plt.rcParams.update({'font.size': 20})

from osgeo import gdal
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.plot import show


def extract_by_mask(shp_path, raster_path, output_path):
    # 
    if not os.path.exists(shp_path):
        print(f"Shapefile not found: {shp_path}")
        return
    if not os.path.exists(raster_path):
        print(f"Raster file not found: {raster_path}")
        return

    # 
    shp = gpd.read_file(shp_path)

    # （）
    geometries = shp.geometry.values

    # 
    with rasterio.open(raster_path) as src:
        # 
        out_image, out_transform = mask(src, geometries, crop=True, nodata=src.nodata)

        # 
        out_meta = src.meta.copy()
        out_meta.update({
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform,
            "nodata": src.nodata,
            "count": out_image.shape[0]  #
        })

        # 
        with rasterio.open(output_path, "w", **out_meta) as dest:
            dest.write(out_image)


def print_geoinfo(dataset, label):
    print(f"{label} - Projection: {dataset.GetProjection()}")
    print(f"{label} - GeoTransform: {dataset.GetGeoTransform()}")


def process_band(dataset, output_image_path):
    if dataset.RasterCount < 2:
        print("Error: Less than 2 bands in input image.")
        return

    # 12
    band1 = dataset.GetRasterBand(1)
    band1_data = band1.ReadAsArray().astype(np.float32)

    band2 = dataset.GetRasterBand(2)
    band2_data = band2.ReadAsArray().astype(np.float32)

    # 
    band_vv = np.sqrt(band1_data ** 2 + band2_data ** 2)

    # 
    driver = gdal.GetDriverByName('GTiff')
    output_dataset = driver.Create(
        output_image_path,
        dataset.RasterXSize,
        dataset.RasterYSize,
        1,
        gdal.GDT_Float32
    )

    if not output_dataset:
        print("Failed to create output image.")
        return

    # 
    output_dataset.SetGeoTransform(dataset.GetGeoTransform())
    output_dataset.SetProjection(dataset.GetProjection())

    # 
    output_band = output_dataset.GetRasterBand(1)
    output_band.WriteArray(band_vv)
    output_band.FlushCache()
    output_dataset.FlushCache()
    output_dataset = None

    # 
    output_dataset = gdal.Open(output_image_path)
    if not output_dataset:
        print("Failed to open output image.")
        return

    print("\nOutput Image Information:")
    print_geoinfo(output_dataset, "Output Image")
    output_dataset = None


if __name__ == "__main__":
    input_image_path = r'D:\yan2\github\2.COSI Corr\2.2 Simulated\T500vvsin_moveextentL8_323216161.tif'
    if not os.path.exists(input_image_path):
        print(f"Input image not found: {input_image_path}")
    else:
        dataset = gdal.Open(input_image_path)
        if not dataset:
            print("Failed to open input image.")
        else:
            print("Input Image Information:")
            print_geoinfo(dataset, "Input Image")

            # 
            vv_output_path = r'D:\yan2\github\3.Comparative Validation\3.1 Error Calculation\T500vvsin_moveextentL8_323216161_vv.tif'
            process_band(dataset, vv_output_path)
            dataset = None


Input Image Information:
Input Image - Projection: PROJCS["UTM_Zone_46N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",93],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["Meter",1],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Input Image - GeoTransform: (718854.299, 15.0, -0.0, 3514673.532, -0.0, -15.0)

Output Image Information:
Output Image - Projection: PROJCS["WGS 84 / UTM zone 46N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",

#，error

In [9]:
#
#time: 2025-09-18
#auther:
import numpy as np
from osgeo import gdal
import sys

def subtract_rasters(input_file1, input_file2, output_file):
    """
     (band1 - band2)
    :
        input_file1: 
        input_file2: 
        output_file: 
    """
    # GDAL
    gdal.AllRegister()
    
    try:
        # 
        ds1 = gdal.Open(input_file1, gdal.GA_ReadOnly)
        if ds1 is None:
            raise IOError(f": {input_file1}")
        
        # 
        ds2 = gdal.Open(input_file2, gdal.GA_ReadOnly)
        if ds2 is None:
            raise IOError(f": {input_file2}")
        
        # 
        band1 = ds1.GetRasterBand(1)
        band2 = ds2.GetRasterBand(1)
        
        # 
        if (ds1.RasterXSize != ds2.RasterXSize) or (ds1.RasterYSize != ds2.RasterYSize):
            raise ValueError("！")
        
        # NumPy
        arr1 = band1.ReadAsArray().astype(np.float32)
        arr2 = band2.ReadAsArray().astype(np.float32)
        
        # NoData
        nodata1 = band1.GetNoDataValue()
        nodata2 = band2.GetNoDataValue()
        
        # 
        result = np.full_like(arr1, -9999.0, dtype=np.float32)
        
        # 
        valid_mask = np.ones_like(arr1, dtype=bool)
        
        if nodata1 is not None:
            valid_mask &= (arr1 != nodata1)
        if nodata2 is not None:
            valid_mask &= (arr2 != nodata2)
        
        #  (arr1 - arr2)
        np.subtract(arr1, arr2, out=result, where=valid_mask)
        
        # NoData
        result[~valid_mask] = -9999.0
        
        # 
        driver = gdal.GetDriverByName('GTiff')
        out_ds = driver.Create(
            output_file,
            ds1.RasterXSize,
            ds1.RasterYSize,
            1,  # 
            gdal.GDT_Float32  # 
        )
        
        # 
        out_ds.SetGeoTransform(ds1.GetGeoTransform())
        out_ds.SetProjection(ds1.GetProjection())
        
        # 
        out_band = out_ds.GetRasterBand(1)
        out_band.WriteArray(result)
        out_band.SetNoDataValue(-9999.0)  # NoData
        out_band.FlushCache()
        
        print(f"！: {output_file}")
        
    except Exception as e:
        print(f": {str(e)}", file=sys.stderr)
        sys.exit(1)
    finally:
        # 
        if 'ds1' in locals(): ds1 = None
        if 'ds2' in locals(): ds2 = None
        if 'out_ds' in locals(): out_ds = None

if __name__ == "__main__":
    # 
    input1 = r'D:\yan2\github\3.Comparative Validation\3.1 Error Calculation\T500vvsin_moveextentL8_323216161_vv.tif'
    input2 =R"D:\yan2\github\1.Simulated image\T500vvsin.tif"
    output =R"D:\yan2\github\3.Comparative Validation\3.1 Error Calculation\error_T500vvsin_moveextentL8_323216161_vv.tif"
    
    subtract_rasters(input1, input2, output)

！: D:\yan2\github\3.Comparative Validation\3.1 Error Calculation\error_T500vvsin_moveextentL8_323216161_vv.tif


#tanfer abs and output pixel value

In [10]:
from osgeo import gdal
import numpy as np
import pandas as pd
import os

def process_raster_with_abs(input_path, output_tif, output_csv):
    """
    ：，TIFFCSV
    
    :
    input_path: TIFF
    output_tif: TIFF
    output_csv: CSV
    """
    try:
        # 
        os.makedirs(os.path.dirname(output_tif), exist_ok=True)
        os.makedirs(os.path.dirname(output_csv), exist_ok=True)
        
        # 
        dataset = gdal.Open(input_path)
        if dataset is None:
            raise Exception(f": {input_path}")
        
        # 
        cols = dataset.RasterXSize
        rows = dataset.RasterYSize
        bands = dataset.RasterCount
        geotransform = dataset.GetGeoTransform()
        projection = dataset.GetProjection()
        
        print(f": {cols} x {rows}, : {bands}")
        
        # 
        abs_bands = []
        for band_idx in range(1, bands + 1):
            band = dataset.GetRasterBand(band_idx)
            pixel_values = band.ReadAsArray()
            abs_pixel_values = np.abs(pixel_values)
            abs_bands.append(abs_pixel_values)
        
        # TIFF
        driver = gdal.GetDriverByName('GTiff')
        out_dataset = driver.Create(
            output_tif, 
            cols, 
            rows, 
            bands, 
            gdal.GDT_Float32  # 
        )
        
        # 
        out_dataset.SetGeoTransform(geotransform)
        out_dataset.SetProjection(projection)
        
        # 
        for band_idx in range(bands):
            out_band = out_dataset.GetRasterBand(band_idx + 1)
            out_band.WriteArray(abs_bands[band_idx])
            out_band.FlushCache()
        
        # 
        out_dataset = None
        print(f"TIFF: {output_tif}")
        
        # 
        x_coords = np.zeros((rows, cols))
        y_coords = np.zeros((rows, cols))
        
        for row in range(rows):
            for col in range(cols):
                x_coords[row, col] = geotransform[0] + col * geotransform[1] + row * geotransform[2]
                y_coords[row, col] = geotransform[3] + col * geotransform[4] + row * geotransform[5]
        
        # 
        data = []
        for row in range(rows):
            for col in range(cols):
                # 
                data.append({
                    'row': row,
                    'col': col,
                    'x_coord': x_coords[row, col],
                    'y_coord': y_coords[row, col],
                    'pixel_value': abs_bands[0][row, col]  # 
                })
        
        df = pd.DataFrame(data)
        
        # CSV
        df.to_csv(output_csv, index=False)
        print(f"CSV: {output_csv}")
        
        # 
        dataset = None
        
        return df, output_tif, output_csv
        
    except Exception as e:
        print(f": {str(e)}")
        return None, None, None

# 
if __name__ == "__main__":
    #   
    input_path = R"D:\yan2\github\3.Comparative Validation\3.1 Error Calculation\T500vvsin_moveextentL8_323216161_vv.tif"
    
    # 
    output_tif = R"D:\yan2\github\3.Comparative Validation\3.1 Error Calculation\abs_error_T500vvsin_moveextentL8_323216161_vv.tif"
    output_csv = r"D:\yan2\github\3.Comparative Validation\3.1 Error Calculation\abs_error_T500vvsin_moveextentL8_323216161_vv.csv"
    
    # CSVTIFF
    df, tif_path, csv_path = process_raster_with_abs(input_path, output_tif, output_csv)
    
    if df is not None:
        print(f"，{len(df)}")
        print("5:")
        print(df.head())

: 1171 x 709, : 1
TIFF: D:\yan2\github\3.Comparative Validation\3.1 Error Calculation\abs_error_T500vvsin_moveextentL8_323216161_vv.tif
CSV: D:\yan2\github\3.Comparative Validation\3.1 Error Calculation\abs_error_T500vvsin_moveextentL8_323216161_vv.csv
，830239
5:
   row  col     x_coord      y_coord  pixel_value
0    0    0  718854.299  3514673.532          NaN
1    0    1  718869.299  3514673.532          NaN
2    0    2  718884.299  3514673.532          NaN
3    0    3  718899.299  3514673.532          NaN
4    0    4  718914.299  3514673.532          NaN


#Simultaneously extract the coordinates of the nodata pixel values for L8 and L9

In [11]:
import pandas as pd
import numpy as np

# CSV
l9_df = pd.read_csv(r'D:\yan2\github\3.Comparative Validation\3.1 Error Calculation\abs_error_T500vvsin_moveextentL9_323216161_vv.csv')
l8_df = pd.read_csv(r'D:\yan2\github\3.Comparative Validation\3.1 Error Calculation\abs_error_T500vvsin_moveextentL8_323216161_vv.csv')

# pixel_value
l9_null_rows = l9_df[l9_df['pixel_value'].isnull()].copy()
l8_null_rows = l8_df[l8_df['pixel_value'].isnull()].copy()

# 
l9_null_rows['source'] = 'L9'
l8_null_rows['source'] = 'L8'

# 
l9_null_rows['coord_id'] = l9_null_rows['x_coord'].astype(str) + '_' + l9_null_rows['y_coord'].astype(str)
l8_null_rows['coord_id'] = l8_null_rows['x_coord'].astype(str) + '_' + l8_null_rows['y_coord'].astype(str)

# 
l9_null_coords = set(l9_null_rows['coord_id'])
l8_null_coords = set(l8_null_rows['coord_id'])

# 
both_null = l9_null_coords & l8_null_coords
only_l9_null = l9_null_coords - l8_null_coords
only_l8_null = l8_null_coords - l9_null_coords

print(f": {len(l9_null_coords | l8_null_coords)}")
print(f"L9L8: {len(both_null)}")
print(f"L9: {len(only_l9_null)}")
print(f"L8: {len(only_l8_null)}")

# 
all_null_rows = pd.concat([l9_null_rows, l8_null_rows], ignore_index=True)

# ，
unique_null_rows = all_null_rows.groupby('coord_id').first().reset_index()

# 
unique_null_rows.drop('coord_id', axis=1, inplace=True)

# CSV
unique_null_rows.to_csv(r'nodata_pixel.csv', index=False, encoding='utf-8-sig')

print(f" {len(unique_null_rows)} ")
print(": .csv")

# 
print("\n5:")
print(unique_null_rows.head())

: 115341
L9L8: 114664
L9: 287
L8: 390
 115341 
: .csv

5:
   row  col     x_coord      y_coord  pixel_value source
0  708    0  718854.299  3504053.532          NaN     L9
1  707    0  718854.299  3504068.532          NaN     L9
2  706    0  718854.299  3504083.532          NaN     L9
3  705    0  718854.299  3504098.532          NaN     L9
4  704    0  718854.299  3504113.532          NaN     L9


At the same time, remove the pixels of the nodata points.

In [ ]:
import rasterio
import pandas as pd
import numpy as np

tif_path = r"D:\yan2\github\3.Comparative Validation\3.1 Error Calculation\abs_error_T500vvsin_moveextentL8_323216161_vv.tif"
csv_path =r"D:\yan2\github\3.Comparative Validation\3.1 Error Calculation\nodata_pixel.csv"
output_path = r"D:\yan2\github\3.Comparative Validation\3.1 Error Calculation\abs_useL9nodata_clipT500errorL8.tif"
nodata_value = np.nan  # NaN
# 
with rasterio.open(tif_path, 'r') as src:
    band = src.read(1)
    profile = src.profile
    height, width = band.shape

# CSV
df = pd.read_csv(csv_path)

# 
assert {'row', 'col'}.issubset(df.columns)
# 
extracted_values = []

for idx, row in df.iterrows():
    r = int(row['row'])
    c = int(row['col'])

    if 0 <= r < height and 0 <= c < width:
        val = band[r, c]
        print(f"({r}, {c}) = {val}")
        extracted_values.append((r, c, val))
        band[r, c] = nodata_value
    else:
        print(f"： ({r}, {c}) ")

# nodata
profile.update(nodata=nodata_value)

# 
with rasterio.open(output_path, 'w', **profile) as dst:
    dst.write(band, 1)

print(f"\n✅ ， {len(extracted_values)} 。")


(708, 0) = nan
(707, 0) = nan
(706, 0) = nan
(705, 0) = nan
(704, 0) = nan
(703, 0) = nan
(702, 0) = nan
(701, 0) = nan
(700, 0) = nan
(699, 0) = nan
(698, 0) = nan
(697, 0) = nan
(696, 0) = nan
(695, 0) = nan
(694, 0) = nan
(693, 0) = nan
(692, 0) = nan
(691, 0) = nan
(690, 0) = nan
(689, 0) = nan
(688, 0) = nan
(687, 0) = nan
(686, 0) = nan
(685, 0) = nan
(684, 0) = nan
(683, 0) = nan
(682, 0) = nan
(681, 0) = nan
(680, 0) = nan
(679, 0) = nan
(678, 0) = nan
(677, 0) = nan
(676, 0) = nan
(675, 0) = nan
(674, 0) = nan
(673, 0) = nan
(672, 0) = nan
(671, 0) = nan
(670, 0) = nan
(669, 0) = nan
(668, 0) = nan
(667, 0) = nan
(666, 0) = nan
(665, 0) = nan
(664, 0) = nan
(663, 0) = nan
(662, 0) = nan
(661, 0) = nan
(660, 0) = nan
(659, 0) = nan
(658, 0) = nan
(657, 0) = nan
(656, 0) = nan
(655, 0) = nan
(654, 0) = nan
(653, 0) = nan
(652, 0) = nan
(651, 0) = nan
(650, 0) = nan
(649, 0) = nan
(648, 0) = nan
(647, 0) = nan
(646, 0) = nan
(645, 0) = nan
(644, 0) = nan
(643, 0) = nan
(642, 0) =

calcution MAE RMSE.....

In [15]:
import numpy as np
from osgeo import gdal
from scipy import stats
path = r"D:\yan1\iceflow\synthetic\moun\3正弦曲线\L9\ERROR\abs_useL8nodata_clipT500errorL9.tif"
# 
def read_tif(path):
    dataset = gdal.Open(path)
    if dataset is None:
        raise FileNotFoundError(f": {path}")
    band = dataset.GetRasterBand(1)
    array = band.ReadAsArray().astype(np.float32)
    nodata = band.GetNoDataValue()
    if nodata is not None:
        array[array == nodata] = np.nan
    return array

# 
error_array = read_tif(path)  # 

# （NaN）
mask = ~np.isnan(error_array)  # 
valid_data = error_array[mask]  # 

# 
if valid_data.size == 0:
    raise ValueError("")

# 
mean = np.nanmean(error_array)
min_val = np.nanmin(error_array)  # min/max
max_val = np.nanmax(error_array)
rmse = np.sqrt(np.nanmean(error_array**2))

# （95%）
confidence = 0.95
n = valid_data.size
sem = stats.sem(valid_data)  # 
h = sem * stats.t.ppf((1 + confidence) / 2., n-1)
ci_lower = mean - h
ci_upper = mean + h

# 
print("="*50)
print(f": {n}")
print(": {:.4f}".format(mean))
print(": {:.4f}".format(max_val))
print(": {:.4f}".format(min_val))
print(" (RMSE): {:.4f}".format(rmse))
print("95%: ({:.4f}, {:.4f})".format(ci_lower, ci_upper))
print("="*50)

: 710770
: 0.6313
: 197.3024
: 0.0000
 (RMSE): 3.3895
95%: (0.6235, 0.6390)


#glacier and rock

In [ ]:
#glacier and rock

import numpy as np
from osgeo import gdal
from scipy import stats

shp_path = r"D:\yan1\iceflow\synthetic\moun\3\shp\ice3.shp"
raster_path = r"\useL9nodata_T500errorL8.tif"
# SHP
def clip_raster_with_shp(raster_path, shp_path):
    """SHP，GDAL（）"""
    # 
    src_ds = gdal.Open(raster_path)
    
    # 
    warp_options = gdal.WarpOptions(
        format='MEM',              # 
        cutlineDSName=shp_path,    # 
        cropToCutline=True,        # 
        dstNodata=np.nan,          # nodataNaN
        resampleAlg=gdal.GRA_NearestNeighbour  # 
    )
    
    # 
    clipped_ds = gdal.Warp('', src_ds, options=warp_options)
    return clipped_ds

def read_clipped_array(ds):
    """GDAL"""
    band = ds.GetRasterBand(1)
    array = band.ReadAsArray().astype(np.float32)
    return array

# 
data_ds = clip_raster_with_shp(raster_path, shp_path)
error_array = read_clipped_array(data_ds)  # 

# （NaN）
mask = ~np.isnan(error_array)  # 
valid_data = error_array[mask]  # 

# 
if valid_data.size == 0:
    raise ValueError("")

# 
mean = np.nanmean(error_array)
min_val = np.nanmin(error_array)  # min/max
max_val = np.nanmax(error_array)
rmse = np.sqrt(np.nanmean(error_array**2))

# （95%）
confidence = 0.95
n = valid_data.size
sem = stats.sem(valid_data)  # 
h = sem * stats.t.ppf((1 + confidence) / 2., n-1)
ci_lower = mean - h
ci_upper = mean + h

# 
print("="*50)
print(f": {n}")
print(": {:.4f}".format(mean))
print(": {:.4f}".format(max_val))
print(": {:.4f}".format(min_val))
print(" (RMSE): {:.4f}".format(rmse))
print("95%: ({:.4f}, {:.4f})".format(ci_lower, ci_upper))
print("="*50)

有效像素数量: 35228
均值: 0.9702
最大值: 209.1588
最小值: -114.4483
均方根误差 (RMSE): 6.2144
95%置信区间: (0.9061, 1.0343)


In [5]:
import numpy as np
from osgeo import gdal

shp_path = r"D:\yan1\iceflow\synthetic\moun\3\shp\clip.shp"
raster_path = r"D:\yan1\iceflow\synthetic\moun\3\L8\COSICORR\clipT500vvsin_moveextentL8_323216161vv.tif"

def clip_raster_with_shp(raster_path, shp_path):
    src_ds = gdal.Open(raster_path)
    
    warp_options = gdal.WarpOptions(
        format='MEM',
        cutlineDSName=shp_path,
        cropToCutline=True,
        dstNodata=np.nan,
        resampleAlg=gdal.GRA_NearestNeighbour
    )
    
    clipped_ds = gdal.Warp('', src_ds, options=warp_options)
    return clipped_ds

def read_clipped_array(ds):
    band = ds.GetRasterBand(1)
    array = band.ReadAsArray().astype(np.float32)
    return array

# 
data_ds = clip_raster_with_shp(raster_path, shp_path)
error_array = read_clipped_array(data_ds)

# =========================
#  NaN 
# =========================
nan_mask = np.isnan(error_array)
nan_count = np.sum(nan_mask)

#  & （）
total_pixels = error_array.size
valid_pixels = np.sum(~nan_mask)

# 
print("="*50)
print(f": {total_pixels}")
print(f"NaN : {nan_count}")
print(f": {valid_pixels}")
print(f"NaN : {nan_count / total_pixels:.4%}")
print("="*50)

总像素数量: 712264
NaN 像素数量: 573
有效像素数量: 711691
NaN 占比: 0.0804%
